# Direct Gas/Vapour Stream MVR Workspace Comparison
This notebook demonstrates the process-component workflow with `PinchWorkspace`. The workspace creates three live `PinchProblem` cases: a baseline case before MVR compression, a dry direct process MVR case, and a direct process MVR case with liquid injection enabled. The cases are then compared on target summaries and grand composite curves.


## Demo case
The small in-memory case has one process zone inside a parent site. `Evaporator vapour` is a near-isothermal hot process vapour stream. It uses `fluid_phase="vapour"` and omits `p_supply`, so OpenPinch derives the inlet pressure from the CoolProp saturation pressure at `t_supply`. The cold `Feed preheat` stream gives the zone a heat load for direct and Total Site targeting.

In [ ]:
import pandas as pd
from IPython.display import display

from OpenPinch import PinchWorkspace
from OpenPinch.lib.enums import ZT

In [ ]:
process_mvr_case = {
    "zone_tree": {
        "name": "Demo Site",
        "type": ZT.S.value,
        "children": [
            {"name": "Evaporation Train", "type": ZT.P.value},
        ],
    },
    "options": {
        "TOP_ZONE_NAME": "Demo Site",
        "TOP_ZONE_IDENTIFIER": ZT.S.value,
        "DO_DIRECT_SITE_TARGETING": True,
    },
    "streams": [
        {
            "zone": "Evaporation Train",
            "name": "Evaporated vapour",
            "t_supply": 99.9,
            "t_target": 100.0,
            "heat_flow": 900.0,
            "dt_cont": 2.0,
            "htc": 1.0,
            "fluid_name": "Water",
            "fluid_phase": "vapour",
        },
        {
            "zone": "Evaporation Train",
            "name": "Evaporator vapour",
            "t_supply": 100.0,
            "t_target": 99.9,
            "heat_flow": 900.0,
            "dt_cont": 2.0,
            "htc": 1.0,
            "fluid_name": "Water",
            "fluid_phase": "vapour",
        },        
        {
            "zone": "Evaporation Train",
            "name": "Feed preheat",
            "t_supply": 40.0,
            "t_target": 105.0,
            "heat_flow": 200.0,
            "dt_cont": 3.0,
            "htc": 1.0,
        },
    ],
    "utilities": [],
}

## Build workspace cases
`PinchWorkspace` keeps the baseline, dry MVR, and liquid-injection MVR cases side by side. Each MVR component is memory-only, so it is added to the live copied `PinchProblem` object. The baseline `before_mvr` problem remains unchanged.


In [ ]:
workspace = PinchWorkspace(
    process_mvr_case,
    project_name="direct_vapour_mvr_demo",
    baseline_name="before_mvr",
)

before_problem = workspace.case("before_mvr")
after_problem = workspace.copy_case("before_mvr", "after_mvr")
after_liquid_injection_problem = workspace.copy_case(
    "before_mvr",
    "after_mvr_liquid_injection",
)

mvr = after_problem.add_component.process_mvr(
    "Evaporator vapour",
    mvr_id="vapour_mvr",
    n_stages=2,
    liquid_injection=False,
    mvr_stage_pressure_ratio=1.25,
)

mvr_liquid_injection = after_liquid_injection_problem.add_component.process_mvr(
    "Evaporator vapour",
    mvr_id="vapour_mvr_liquid_injection",
    n_stages=2,
    liquid_injection=True,
    mvr_stage_pressure_ratio=1.25,
)

component_rows = [
    {
        "case": "before_mvr",
        "has_mvr_component": False,
        "process_components": len(before_problem.process_components),
    }
]

for case_name, problem, component in [
    ("after_mvr", after_problem, mvr),
    (
        "after_mvr_liquid_injection",
        after_liquid_injection_problem,
        mvr_liquid_injection,
    ),
]:
    stages = component.stage_results_by_state["0"]
    component_rows.append(
        {
            "case": case_name,
            "has_mvr_component": True,
            "process_components": len(problem.process_components),
            "mvr_id": component.id,
            "liquid_injection": component.settings.liquid_injection,
            "liquid_injection_applied": any(
                stage.liquid_injection_applied for stage in stages
            ),
            "liquid_injection_heat_kW": sum(
                stage.q_liquid_injection for stage in stages
            ),
            "derived_source_pressure_kPa": float(component.original_streams[0].p_supply),
            "replacement_stream_count": len(component.replacement_streams),
            "stage_pressure_ratio": component.settings.mvr_stage_pressure_ratio,
            "total_stage_work_kW": sum(stage.work for stage in stages),
        }
    )

pd.DataFrame(component_rows)


## Solve all cases
Run the same targeting sequence for every `PinchProblem` object: local direct heat integration for the process zone, direct targeting for subzones, and then Total Site targeting on the parent zone.


In [ ]:
for case_name in workspace.list_cases():
    problem = workspace.case(case_name)
    problem.target.direct_heat_integration(zone_name="Evaporation Train")
    problem.target.direct_heat_integration(include_subzones=True)
    problem.target.indirect_heat_integration()

pd.concat(
    {
        case_name: workspace.summary_frame(case_name=case_name)
        for case_name in workspace.list_cases()
    },
    names=["case", "row"],
)

## Compare targets
`workspace.compare_cases(...)` compares solved target rows between live `PinchProblem` cases. The detailed summaries expose the process-component compressor work added to each MVR case.


In [ ]:
target_comparisons = pd.concat(
    {
        "dry_mvr": workspace.compare_cases(
            "before_mvr",
            "after_mvr",
            target_name="Evaporation Train/Direct Integration",
        ),
        "liquid_injection_mvr": workspace.compare_cases(
            "before_mvr",
            "after_mvr_liquid_injection",
            target_name="Evaporation Train/Direct Integration",
        ),
    },
    names=["comparison", "row"],
)

detailed_summaries = pd.concat(
    {
        case_name: workspace.summary_frame(case_name=case_name, detailed=True)
        for case_name in workspace.list_cases()
    },
    names=["case", "row"],
)

display(target_comparisons)

detailed_summaries[
    [
        "Target",
        "Qh (value)",
        "Qc (value)",
        "Qr (value)",
        "Work Target (value)",
        "Process Component Work (value)",
    ]
]


## Inspect replacement streams
The original stream remains in each MVR case hot stream collection with `active=False`. The active replacement streams are the high-pressure piecewise-linear hot segments generated from the compressed vapour cooling curve.


In [ ]:
stream_rows = []
for case_name, problem in [
    ("after_mvr", after_problem),
    ("after_mvr_liquid_injection", after_liquid_injection_problem),
]:
    zone = problem.master_zone.get_subzone("Evaporation Train")
    for key, stream in zone.hot_streams.items():
        stream_rows.append(
            {
                "case": case_name,
                "collection_key": key,
                "stream_name": stream.name,
                "active": stream.active,
                "t_supply_degC": float(stream.t_supply),
                "t_target_degC": float(stream.t_target),
                "p_supply_kPa": float(stream.p_supply or 0.0),
                "p_target_kPa": float(stream.p_target or 0.0),
                "heat_flow_kW": float(stream.heat_flow),
            }
        )

pd.DataFrame(stream_rows)


## Compare grand composite curves
Each case exposes the normal plot accessor. The raw GCC payloads are useful for tabular comparison, while the Plotly figures show the profile shift visually.

In [ ]:
gcc_payloads = {
    case_name: workspace.case(case_name).plot.grand_composite_curve(
        zone_name="Evaporation Train",
        return_graph_data=True,
    )
    for case_name in workspace.list_cases()
}

gcc_rows = []
for case_name, payload in gcc_payloads.items():
    for segment_index, segment in enumerate(payload["segments"], start=1):
        heat_loads = [point["x"] for point in segment["data_points"]]
        temperatures = [point["y"] for point in segment["data_points"]]
        gcc_rows.append(
            {
                "case": case_name,
                "segment": segment_index,
                "title": segment["title"],
                "points": len(segment["data_points"]),
                "min_heat_load_kW": min(heat_loads),
                "max_heat_load_kW": max(heat_loads),
                "min_temperature_degC": min(temperatures),
                "max_temperature_degC": max(temperatures),
            }
        )

pd.DataFrame(gcc_rows)

In [ ]:
before_gcc = before_problem.plot.grand_composite_curve(zone_name="Evaporation Train")
after_gcc = after_problem.plot.grand_composite_curve(zone_name="Evaporation Train")
after_liquid_injection_gcc = (
    after_liquid_injection_problem.plot.grand_composite_curve(
        zone_name="Evaporation Train"
    )
)

display(before_gcc)
display(after_gcc)
display(after_liquid_injection_gcc)


## Toggle the dry MVR case
The component can still be toggled on the `after_mvr` problem. Deactivation restores the original stream and invalidates stale targets for that case only; the separate liquid-injection case remains active.


In [ ]:
mvr.deactivate()
restored_target = after_problem.target.direct_heat_integration(
    zone_name="Evaporation Train"
)

mvr.activate()
reapplied_target = after_problem.target.direct_heat_integration(
    zone_name="Evaporation Train"
)

pd.DataFrame(
    [
        {
            "state": "dry_deactivated",
            "component_active": mvr.active,
            "liquid_injection_case_active": mvr_liquid_injection.active,
            "original_active": all(stream.active for stream in mvr.original_streams),
            "replacement_active": any(stream.active for stream in mvr.replacement_streams),
            "heat_recovery_kW": restored_target.heat_recovery_target,
            "process_component_work_kW": restored_target.process_component_work_target,
        },
        {
            "state": "dry_reactivated",
            "component_active": mvr.active,
            "liquid_injection_case_active": mvr_liquid_injection.active,
            "original_active": any(stream.active for stream in mvr.original_streams),
            "replacement_active": all(stream.active for stream in mvr.replacement_streams),
            "heat_recovery_kW": reapplied_target.heat_recovery_target,
            "process_component_work_kW": reapplied_target.process_component_work_target,
        },
    ]
)
